<a href="https://colab.research.google.com/github/ademirsantosjr/cotahist-google-colab/blob/main/cotahist_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COTAHIST Processing

## Drive Mounting

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

Mounted at /content/drive


## Directories Definition

In [2]:
# Define and access the root of your project
ROOT_DIR = '/content/drive/MyDrive/3 - Projects/B3_Analyses'

if os.path.exists(ROOT_DIR):
    os.chdir(ROOT_DIR)

    # 3. Create required data directories if they don't exist
    os.makedirs('./data/input', exist_ok=True)
    os.makedirs('./data/output', exist_ok=True)

    print(f"Project root set to: {os.getcwd()}")
else:
    print("CRITICAL ERROR: ROOT_DIR not found. Check your folder path in Google Drive.")

Project root set to: /content/drive/MyDrive/3 - Projects/B3_Analyses


## Extraction Logic

In [3]:
import pandas as pd
import zipfile
import os
from pathlib import Path

def process_zipped_b3_file(zip_path):
    """
    Reads B3 COTAHIST ZIP files, filters only detail records (01),
    and renames columns according to official B3 specifications (e.g., CODNEG).
    """

    # 1. Official B3 Column Specifications (245 bytes layout)
    specs = [
        (0, 2), (2, 10), (10, 12), (12, 24), (24, 27), (27, 39), (39, 49),
        (49, 52), (52, 56), (56, 69), (69, 82), (82, 95), (95, 108),
        (108, 121), (121, 134), (134, 147), (147, 152), (152, 170),
        (170, 188), (188, 201), (201, 202), (202, 210), (210, 217),
        (217, 230), (230, 242), (242, 245)
    ]

    # Column names matching official B3 technical acronyms
    col_names = [
        'TIPREG', 'DATPRG', 'CODBDI', 'CODNEG', 'TPMERC',
        'NOMRES', 'ESPECI', 'PRAZOT', 'MODREF', 'PREABE',
        'PREMAX', 'PREMIN', 'PREMED', 'PREULT', 'PREOFC',
        'PREOFV', 'TOTNEG', 'QUATOT', 'VOLTOT', 'PREEXE',
        'INDOPC', 'DATVEN', 'FATCOT', 'PTOEXE', 'CODISI', 'DISMES'
    ]

    print(f"Processing ZIP: {os.path.basename(zip_path)}")

    with zipfile.ZipFile(zip_path, 'r') as z:
        # Find the TXT file inside the ZIP
        txt_filename = [f for f in z.namelist() if f.lower().endswith('.txt')][0]
        with z.open(txt_filename) as f:
            # Load raw fixed-width data
            df = pd.read_fwf(f, colspecs=specs, names=col_names, encoding='latin1')

    # 2. Filter: Keep only Detail Records (Type 01)
    # This automatically discards Header (00) and Trailer (99)
    df = df[df['TIPREG'] == 1].copy()

    # 3. Numeric Conversion and Decimal Adjustment (V99 and V06 formats)
    # Most price fields are (11)V99 (divide by 100)
    v99_cols = ['PREABE', 'PREMAX', 'PREMIN', 'PREMED', 'PREULT',
                'PREOFC', 'PREOFV', 'VOLTOT', 'PREEXE']

    for col in v99_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce') / 100.0

    # PTOEXE is usually (11)V06 according to some B3 manuals,
    # but we'll stick to numeric conversion for safety
    df['PTOEXE'] = pd.to_numeric(df['PTOEXE'], errors='coerce')

    # Convert CODBDI to int64, as it should not have decimal values and appears to be fully populated.
    df['CODBDI'] = df['CODBDI'].astype('int64')

    # Convert date columns to datetime objects
    # DATPRG is already a string in YYYYMMDD format
    df['DATPRG'] = pd.to_datetime(df['DATPRG'], format='%Y%m%d', errors='coerce')
    # DATVEN might be float initially, convert to string first, then to datetime
    df['DATVEN'] = pd.to_datetime(df['DATVEN'], format='%Y%m%d', errors='coerce')

    # 4. String Sanitization
    # Strip whitespace from all string columns (CODNEG, NOMRES, etc.)
    for col in df.select_dtypes(['object']).columns:
        df[col] = df[col].astype(str).str.strip()

    # 5. Export to Parquet
    output_filename = os.path.basename(zip_path).lower().replace('.zip', '.parquet')
    output_path = os.path.join('./data/output', output_filename)

    df.to_parquet(output_path, index=False)
    print(f"Success! Saved as {output_path}")

# --- Automation Loop ---
INPUT_DIR = './data/input'
for zip_file in Path(INPUT_DIR).glob('*.ZIP'):
    # Check if parquet already exists to skip re-processing
    expected_parquet = str(zip_file).lower().replace('.zip', '.parquet').replace('input', 'output')
    if not os.path.exists(expected_parquet):
        process_zipped_b3_file(str(zip_file))

Processing ZIP: COTAHIST_A2025.ZIP
Success! Saved as ./data/output/cotahist_a2025.parquet


## Data Exploration of Parquet Files

In [4]:
import pandas as pd
import os

# Assuming you have at least one parquet file in the output directory
output_files = [f for f in os.listdir('./data/output') if f.endswith('.parquet')]

if output_files:
    # Take the first parquet file found for exploration
    parquet_file_path = os.path.join('./data/output', output_files[0])
    print(f"Exploring data from: {parquet_file_path}")

    # Load the Parquet file into a pandas DataFrame
    df_parquet = pd.read_parquet(parquet_file_path)

    print("\nFirst 5 rows of the DataFrame:")
    display(df_parquet.head())

    print("\nDataFrame Info (datatypes and non-null counts):")
    df_parquet.info()

    print("\nDescriptive statistics for numerical columns:")
    display(df_parquet.describe(include='all'))
else:
    print("No Parquet files found in ./data/output. Please ensure the conversion process ran successfully.")

Exploring data from: ./data/output/cotahist_a2025.parquet

First 5 rows of the DataFrame:


,TIPREG,DATPRG,CODBDI,CODNEG,TPMERC,NOMRES,ESPECI,PRAZOT,MODREF,PREABE,...,TOTNEG,QUATOT,VOLTOT,PREEXE,INDOPC,DATVEN,FATCOT,PTOEXE,CODISI,DISMES
0,1,2025-01-20,12,ITIP11,10,FII INTER IP,CI,NaN,R$,69.00,...,59.0,1120.0,76847.5,0.0,0.0,NaT,1.0,0.0,BRITIPCTF002,136.0
1,1,2025-01-20,12,ITIT11,10,FII INTER IT,CI,NaN,R$,67.05,...,43.0,1680.0,112954.4,0.0,0.0,NaT,1.0,0.0,BRITITCTF004,136.0
2,1,2025-01-20,34,ITLC34,10,INTEL,DRN,NaN,R$,21.52,...,2020.0,145530.0,3270594.4,0.0,0.0,NaT,1.0,0.0,BRITLCBDR005,156.0
3,1,2025-01-20,12,ITRI11,10,FII ITRI,CI,NaN,R$,66.65,...,2518.0,17540.0,1170220.9,0.0,0.0,NaT,1.0,0.0,BRITRICTF008,109.0
4,1,2025-01-20,2,ITSA3,10,ITAUSA,ON N1,NaN,R$,9.20,...,372.0,98000.0,907978.0,0.0,0.0,NaT,1.0,0.0,BRITSAACNOR0,446.0



DataFrame Info (datatypes and non-null counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3174698 entries, 0 to 3174697
Data columns (total 26 columns):
 #   Column  Dtype         
---  ------  -----         
 0   TIPREG  int64         
 1   DATPRG  datetime64[ns]
 2   CODBDI  int64         
 3   CODNEG  object        
 4   TPMERC  int64         
 5   NOMRES  object        
 6   ESPECI  object        
 7   PRAZOT  float64       
 8   MODREF  object        
 9   PREABE  float64       
 10  PREMAX  float64       
 11  PREMIN  float64       
 12  PREMED  float64       
 13  PREULT  float64       
 14  PREOFC  float64       
 15  PREOFV  float64       
 16  TOTNEG  float64       
 17  QUATOT  float64       
 18  VOLTOT  float64       
 19  PREEXE  float64       
 20  INDOPC  float64       
 21  DATVEN  datetime64[ns]
 22  FATCOT  float64       
 23  PTOEXE  float64       
 24  CODISI  object        
 25  DISMES  float64       
dtypes: datetime64[ns](2), float64(16), int64(3), obje

,TIPREG,DATPRG,CODBDI,CODNEG,TPMERC,NOMRES,ESPECI,PRAZOT,MODREF,PREABE,...,TOTNEG,QUATOT,VOLTOT,PREEXE,INDOPC,DATVEN,FATCOT,PTOEXE,CODISI,DISMES
count,3174698.0,3174698,3.174698e+06,3174698,3.174698e+06,3174698,3174698,2.738110e+06,3174698,3.174698e+06,...,3.174698e+06,3.174698e+06,3.174698e+06,3.174698e+06,3174698.0,2698544,3.174698e+06,3.174698e+06,3174698,3.174698e+06
unique,NaN,NaN,NaN,247484,NaN,3726,230,NaN,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2447,NaN
top,NaN,NaN,NaN,PETR4T,NaN,PETRE,ON NM,NaN,R$,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BRPETRACNPR6,NaN
freq,NaN,NaN,NaN,1668,NaN,90175,1709500,NaN,3174698,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,160288,NaN
mean,1.0,2025-07-09 03:37:04.889171456,7.291604e+01,NaN,6.461804e+01,NaN,NaN,1.223337e+00,NaN,8.425271e+01,...,2.800655e+02,7.749644e+05,1.890733e+06,1.399741e+03,0.0,2025-09-02 02:28:10.202124288,2.508334e+00,1.378953e+09,NaN,1.734166e+02
min,1.0,2025-01-02 00:00:00,2.000000e+00,NaN,1.000000e+01,NaN,NaN,0.000000e+00,NaN,0.000000e+00,...,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,2025-01-03 00:00:00,1.000000e+00,0.000000e+00,NaN,2.000000e+00
25%,1.0,2025-04-09 00:00:00,7.800000e+01,NaN,7.000000e+01,NaN,NaN,0.000000e+00,NaN,2.000000e-01,...,2.000000e+00,6.000000e+02,4.010000e+02,6.120000e+00,0.0,2025-05-16 00:00:00,1.000000e+00,0.000000e+00,NaN,1.110000e+02
50%,1.0,2025-07-15 00:00:00,7.800000e+01,NaN,7.000000e+01,NaN,NaN,0.000000e+00,NaN,9.000000e-01,...,4.000000e+00,3.900000e+03,3.419000e+03,1.898000e+01,0.0,2025-08-15 00:00:00,1.000000e+00,0.000000e+00,NaN,1.280000e+02
75%,1.0,2025-10-09 00:00:00,8.200000e+01,NaN,8.000000e+01,NaN,NaN,0.000000e+00,NaN,4.360000e+00,...,1.700000e+01,2.190000e+04,2.952700e+04,3.888000e+01,0.0,2025-11-21 00:00:00,1.000000e+00,0.000000e+00,NaN,2.060000e+02
max,1.0,2025-12-30 00:00:00,9.600000e+01,NaN,8.000000e+01,NaN,NaN,9.980000e+02,NaN,4.000000e+06,...,9.950100e+04,9.395935e+11,2.206794e+10,5.000000e+05,0.0,2027-12-17 00:00:00,1.000000e+04,5.000000e+11,NaN,7.950000e+02


## Stock Options

- Types of Market (TPMERC) that represents options:
  - 70 => CALL
  - 80 => PUT

### Listing `ESPECI` for Options (CALL and PUT)

In [5]:
# Filter the DataFrame for option market types (CALL=70, PUT=80)
options_df = df_parquet[df_parquet['CODBDI'].isin([78, 82])]

# List the distinct values of the 'NOMRES' column for options
distinct_option_names = options_df['NOMRES'].unique()

print("Distinct 'NOMRES' values for the options market (CALL/PUT):")
for option_name in distinct_option_names:
    print(option_name)

Distinct 'NOMRES' values for the options market (CALL/PUT):
AAPL
AAPL    /ED
ABCBE
ABCBE    /EJ
ABCB    /EJ
ABCB
ABEV  FM/EDJ
ABEV  FM
ABEV
ABEV    /EDJ
ABEVE    /ED
ABEVE
ABEV  FM/ED
ABEV    /ED
ABEVE  FM/ED
ABEVE  FM
AERIE
AERI
AGROE
AGRO
ALOSE
ALOSE    /ED
ALOSE    /EJ
ALOS    /EJ
ALOS
ALOS    /ED
ALOS  FM/ED
ALOS  FM
ALOS  FM/EJ
ALOSE  FM
ALOSE  FM/ED
ALOSE  FM/EJ
ALPA
ALPAE
ALPA  FM
ALPA    /EDJ
ALPA  FM/EDJ
ALPAE  FM
ALPAE    /ED
ALPAE  FM/ED
ALUPE
ALUP
ALUP    /EDB
ALUP    /ED
ALUPE    /ED
AMBP    /ATZ
AMBP
AMBPE
AMAR
ANIM
ANIME
ANIME    /ED
ANIM    /ED
ANIM  FM
AMARE
ANIME  FM
AMBPE    /AT
ARML    /EJ
ARML
ARMLE
AMZO
ASAI
ASAI    /EJ
ASAI    /ED
AMZOE
ASAIE    /EJ
ASAIE
ASAIE  FM
ASAIE  FM/EJ
ASAIE    /ED
ASAI  FM
ASAI  FM/EJ
ARMLE    /EJ
ASAIE  FM/ED
AURE
AUREE    /ED
AUREE
ASAI  FM/ED
AURE    /ED
AZULE
AZULE    /ES
AZUL
AZUL    /ES
AZUL  FM
AZUL  FM/ES
AZULE  FM
AZULE  FM/ES
AZZA  FM
AZZA
AZZASE 21
AZZAE
B3SA  FM/EJ
B3SA  FM
AZZAE  FM
B3SAE    /EJ
B3SAE
B3SA
B3SA    /EJ
B3SAE

### Function to List Options for a Specific Asset

This `list_specific_options` function filters the options DataFrame (`df_parquet`) based on:

- **`nomres_prefix`**: A prefix for the `NOMRES` (Asset's Short Name) column. For example, 'PETR' for Petrobras shares.
- **`especi_prefix`**: A prefix for the `ESPECI` (Asset Specification) column. For example, 'ON' for ordinary shares.
- **`option_type`**: The type of option to filter. It can be `'CALL'` (70), `'PUT'` (80), or `'BOTH'` (both).

In [6]:
def list_specific_options(df, nomres_prefix=None, especi_prefix=None, option_type='BOTH'):
    """
    Filters the options DataFrame based on the asset's name (NOMRES),
    specification (ESPECI), and option type (CALL, PUT, or BOTH).

    Args:
        df (pd.DataFrame): The DataFrame containing COTAHIST data.
        nomres_prefix (str, optional): Prefix to filter the 'NOMRES' column.
                                       Defaults to None.
        especi_prefix (str, optional): Prefix to filter the 'ESPECI' column.
                                       Defaults to None.
        option_type (str): 'CALL', 'PUT', or 'BOTH'. Defaults to 'BOTH'.

    Returns:
        pd.DataFrame: Filtered DataFrame with corresponding options.
    """

    # 1. Filter by option market types (CODBDI)
    if option_type == 'CALL':
        filtered_df = df[df['CODBDI'] == 78].copy() # 78 for CALL
    elif option_type == 'PUT':
        filtered_df = df[df['CODBDI'] == 82].copy()  # 82 for PUT
    elif option_type == 'BOTH':
        filtered_df = df[df['CODBDI'].isin([78, 82])].copy()
    else:
        raise ValueError("option_type must be 'CALL', 'PUT', or 'BOTH'.")

    # 2. Filter by NOMRES (Asset's Short Name)
    if nomres_prefix:
        filtered_df = filtered_df[filtered_df['NOMRES'].str.startswith(nomres_prefix, na=False)]

    # 3. Filter by ESPECI (Asset Specification)
    if especi_prefix:
        filtered_df = filtered_df[filtered_df['ESPECI'].str.startswith(especi_prefix, na=False)]

    return filtered_df.sort_values(by=['DATPRG', 'CODNEG'])

print("Function 'list_specific_options' defined successfully!")

Function 'list_specific_options' defined successfully!


### Usage Example: CALL Options for PETR (Petrobras) of type 'ON'

In [7]:
# Example: List CALL options for Petrobras (NOMRES starting with 'PETR') with ESPECI starting with 'ON'
petr_on_put_options = list_specific_options(df_parquet,
                                      nomres_prefix='PETR',
                                      especi_prefix='ON',
                                      option_type='PUT')

print(f"Total 'PETR' with 'ON' PUT options found: {len(petr_on_put_options)}")
display(petr_on_put_options.head())

Total 'PETR' with 'ON' PUT options found: 5885


,TIPREG,DATPRG,CODBDI,CODNEG,TPMERC,NOMRES,ESPECI,PRAZOT,MODREF,PREABE,...,TOTNEG,QUATOT,VOLTOT,PREEXE,INDOPC,DATVEN,FATCOT,PTOEXE,CODISI,DISMES
967881,1,2025-01-02,82,PETRM36,80,PETRE /ED,ON N2,0.0,R$,0.03,...,2.0,100000.0,3000.0,34.07,0.0,2025-01-17,1.0,0.0,BRPETRACNOR9,205.0
970939,1,2025-01-02,82,PETRM366,80,PETRE /ED,ON N2,0.0,R$,0.02,...,1.0,100.0,2.0,33.82,0.0,2025-01-17,1.0,0.0,BRPETRACNOR9,206.0
970462,1,2025-01-02,82,PETRM402,80,PETRE /ED,ON N2,0.0,R$,0.11,...,104.0,69900.0,7450.0,37.57,0.0,2025-01-17,1.0,0.0,BRPETRACNOR9,206.0
965607,1,2025-01-02,82,PETRM403,80,PETRE /ED,ON N2,0.0,R$,0.11,...,1.0,100.0,11.0,36.57,0.0,2025-01-17,1.0,0.0,BRPETRACNOR9,205.0
965610,1,2025-01-02,82,PETRM408,80,PETRE /ED,ON N2,0.0,R$,0.12,...,76.0,74500.0,9120.0,37.07,0.0,2025-01-17,1.0,0.0,BRPETRACNOR9,205.0
